---  
 
## Before Beginning

- Yesterday, you directly wrote the code w = w - lr * w.grad, right? optimizer.step() is like a magical line that does that for you.

#### Training Loop
- The structure of the training loop is used almost as is in the CNN and RNN models you will learn in the future.
- The following pseudo-code is the learning form of most artificial intelligence.

In [ ]:
# 1. Define model, loss function, optimizer
model = MyModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 2. Prepare DataLoader
dataloader = DataLoader(...)

# 3. Training Loop (Repeat N epochs)
for epoch in range(num_epochs):
    # Get mini-batch from DataLoader
    for data, labels in dataloader:
        # 3-1. Initialize Gradients
        optimizer.zero_grad()

        # 3-2. Forward Pass
        outputs = model(data)

        # 3-3. Calculate Loss
        loss = criterion(outputs, labels)

        # 3-4. Backward Pass
        loss.backward()

        # 3-5. Update Parameters
        optimizer.step()

    print(f'Epoch {epoch+1}, Loss: {loss.item()}')

#### Common Mistakes

- Missing optimizer.zero_grad(): "Why do I need to zero the gradients every time?"
- Because PyTorch 'accumulates' gradients instead of 'overwriting' them when calling backward()

- Tensor Device Mismatch (CPU vs GPU):
    * Errors occurring because the model is on the GPU but the data remains on the CPU are very common.
    * You must send the data to the same device as the model within the training loop, like data = data.to(device), labels = labels.to(device).

- Switching the Model's Evaluation Mode:
    * It's not essential, but a brief introduction to the concepts of model.train() and model.eval()
    * It's not very important now, but from Day 5 onwards, there are cases where the model's behavior differs between training and evaluation.
    * It's good to get into the habit of switching between these two modes from now on.  
  
  
---


## Day 4 Advanced Supplementary Learning: 
### Mastering Building a Real Neural Network (Revised Edition)

#### Learning Objective: To gain the ability to build a complete training pipeline in a reusable structure by automating repetitive tasks using PyTorch's high-level APIs (nn.Module, Optimizer, DataLoader).

### Concept Check Quiz
This quiz will thoroughly check your understanding of the core concepts.

1. What are the respective roles of the __init__ and forward methods in an nn.Module class?

2. What is the role of an Optimizer, and at which stage of the training loop is it used?

3. What are the respective roles of Dataset and DataLoader, and why should they be used together?

4. List the three core steps of the training loop (optimizer.zero_grad(), loss.backward(), optimizer.step()) in the correct order and explain the role of each step.

5. What does the code nn.Linear(in_features=10, out_features=5) mean? What are the input and output sizes of this layer?

6. What is the term for one full pass through the entire dataset during training? And what does the batch_size in a DataLoader signify?

7. Why do we pass model.parameters() to the optimizer? What would happen if this code were omitted?

8. Name one loss function typically used for regression problems and one for classification problems in PyTorch, and briefly explain their difference.

9. What is the purpose of the loss.item() code? What is the difference if you print the loss tensor itself without .item()?

10. Why do we use model.train() mode for training and model.eval() mode for evaluation? (Hint: Dropout, BatchNorm, etc.)

11. In the code torch.arange(100).view(-1, 1), what role does .view(-1, 1) play?

12. What is the main reason for using activation functions like nn.ReLU in a neural network model? What limitations would a model have without activation functions?

13. While training, the loss value does not decrease at all or even diverges. What hyperparameter should be suspected first, and how should it be adjusted?

14. What are the conceptual differences between the SGD and Adam optimizers? Which one generally tends to converge faster?

15. What should the __getitem__ method of a CustomDataset class return, and in what data type?

16. When loss.backward() is called, what value is stored in the .grad attribute of a tensor that was set with requires_grad=False?

### Code Walkthroughs and Exercises  
  

Increase your adaptability to the PyTorch pipeline structure by following code examples for various scenarios and immediately solving related exercises.

- **Topic 1: Basic Linear Regression (Review)**
    - Scenario: Create the most basic regression model that predicts a single output (y) from a single input (x).
    - Core Concepts: nn.Linear(1, 1), nn.MSELoss

In [ ]:
# [Topic 1] Core Code
# 0. Import necessary libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# 1. Prepare data: y = 3x + 5
X1 = torch.arange(1, 101, 1, dtype=torch.float32).view(-1, 1)
y1 = 3 * X1 + 5 + torch.randn(100, 1) * 2

# 2. Define Dataset and DataLoader
class CustomDataset(Dataset):
    def __init__(self, x, y): self.x, self.y = x, y
    def __getitem__(self, i): return (self.x[i], self.y[i])
    def __len__(self): return len(self.x)

train_dataset1 = CustomDataset(X1, y1)
train_loader1 = DataLoader(train_dataset1, batch_size=10, shuffle=True)

# 3. Define model, loss function, and optimizer
model1 = nn.Linear(in_features=1, out_features=1)
criterion1 = nn.MSELoss()
optimizer1 = torch.optim.SGD(model1.parameters(), lr=0.01)

# 4. Training loop
print("--- Starting Basic Linear Regression Training ---")
for epoch in range(50):
    for inputs, labels in train_loader1:
        outputs = model1(inputs)
        loss = criterion1(outputs, labels)
        optimizer1.zero_grad()
        loss.backward()
        optimizer1.step()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/50], Loss: {loss.item():.4f}')

- **[Topic 1] Programming Exercises**  
    - **Problem 1-1 (Optimizer Swap & Comparison)**: Change the optimizer in the code above from torch.optim.SGD to torch.optim.Adam and retrain the model. With the same learning rate and number of epochs, which optimizer, SGD or Adam, reduces the loss faster?
    - **Problem 1-2 (Visualize Training Process)**: Store the calculated loss value for each epoch in a Python list. After training is complete, use Matplotlib to plot a line graph showing the trend of loss reduction against the epochs.  
    ---

- **Topic 2: Multiple Linear Regression**
    - Scenario: Create a model that predicts a single output (y) from two inputs (x1, x2).
    - Core Concept: nn.Linear(2, 1). This handles the situation where the number of input features increases from one to two.
